# 2. Data Preprocessing
## Digital Twin - Data Integration Project

**Objective**: Clean and prepare data for modeling (using EDA insights)  
**Input**: Raw datasets from `../Data/` folder  
**Output**: Clean, standardized, integrated dataset

---

### Preprocessing Tasks:
1. Load raw datasets
2. Schema standardization (unified format)
3. Handle outliers
4. Handle missing values
5. Feature engineering
6. Temporal sequence synthesis
7. Normalization & scaling
8. Categorical encoding
9. Export processed datasets


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import KNNImputer
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Set Data Paths

In [2]:
# Paths
DATA_DIR = Path('../Data')
OUTPUT_DIR = Path('../Data_preprocessing/output')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# Dataset file names
DATA_FILES = {
    'heart_indicators': DATA_DIR / 'heart_disease_health_indicators_BRFSS2015.csv',
    'cardiovascular': DATA_DIR / 'cardio_train.csv',
    'diabetes_indicators': DATA_DIR / 'diabetes_binary_health_indicators_BRFSS2015.csv',
}

print(f"Output directory: {OUTPUT_DIR}")
print("Paths configured")

Output directory: ../Data_preprocessing/output
Paths configured


## 3. Load Raw Datasets

In [3]:
datasets = {}

for name, path in DATA_FILES.items():
    try:
        if path.name == 'cardio_train.csv':
            df = pd.read_csv(path, sep=';')
        else:
            df = pd.read_csv(path)

        datasets[name] = df
        print(f"✓ Loaded {name}: {df.shape}")
    except FileNotFoundError:
        print(f"✗ File not found: {path}")
    except Exception as e:
        print(f"✗ Error: {str(e)}")

✓ Loaded heart_indicators: (253680, 22)
✓ Loaded cardiovascular: (70000, 13)
✓ Loaded diabetes_indicators: (253680, 22)


## 4. Define Common Schema

In [4]:
COMMON_SCHEMA = {
    'patient_id': 'str',
    'age': 'int',
    'sex': 'str',
    'blood_pressure_systolic': 'float',
    'blood_pressure_diastolic': 'float',
    'bmi': 'float',
    'cholesterol': 'float',
    'glucose': 'float',
    'physical_activity': 'int',
    'smoking_status': 'str',
    'alcohol_consumption': 'int',
    'medical_history_diabetes': 'int',
    'medical_history_hypertension': 'int',
    'medical_history_heart_disease': 'int',
    'target_disease': 'int',
    'data_source': 'str',
    'collection_date': 'datetime'
}

print(f"Common schema defined with {len(COMMON_SCHEMA)} features")

Common schema defined with 17 features


## 5. Schema Standardization (Mapping)

In [10]:
def standardize_heart_indicators(df):
    """Map Heart Disease Health Indicators to common schema"""
    df_std = pd.DataFrame()
    df_std['patient_id'] = [f"HI_{i}" for i in range(len(df))]
    df_std['age'] = (df['Age'] * 5).astype(int)  # Age is encoded in 5-year bins
    df_std['sex'] = df['Sex'].map({1: 'Female', 2: 'Male'})
    df_std['bmi'] = df['BMI'].astype(float) if 'BMI' in df.columns else np.nan
    df_std['physical_activity'] = df['PhysActivity'].astype(int)
    df_std['diet_quality'] = df['Fruits'].astype(int) + df['Veggies'].astype(int)
    df_std['smoking_status'] = df['Smoker'].map({1: 'Current', 0: 'Never'})
    df_std['alcohol_consumption'] = df['HvyAlcoholConsump'].astype(int)
    df_std['medical_history_diabetes'] = df['Diabetes'].astype(int)
    df_std['medical_history_hypertension'] = df['HighBP'].astype(int)
    df_std['medical_history_heart_disease'] = df['HeartDiseaseorAttack'].astype(int)
    df_std['target_disease'] = df['HeartDiseaseorAttack'].astype(int)
    df_std['data_source'] = 'Heart_Indicators'
    df_std['collection_date'] = pd.Timestamp('2015-01-01')
    return df_std

In [8]:
def standardize_cardiovascular(df):
    """Map Cardiovascular Disease dataset to common schema"""
    df_std = pd.DataFrame()
    df_std['patient_id'] = [f"CV_{i}" for i in range(len(df))]
    df_std['age'] = (df['age'] / 365).astype(int)
    df_std['sex'] = df['gender'].map({1: 'Female', 2: 'Male'})
    df_std['blood_pressure_systolic'] = df['ap_hi'].astype(float)
    df_std['blood_pressure_diastolic'] = df['ap_lo'].astype(float)
    df_std['cholesterol'] = df['cholesterol'].astype(float)
    df_std['glucose'] = df['gluc'].astype(float)
    df_std['smoking_status'] = df['smoke'].map({1: 'Current', 0: 'Never'})
    df_std['alcohol_consumption'] = df['alco'].astype(int)
    df_std['physical_activity'] = df['active'].astype(int)
    df_std['target_disease'] = df['cardio'].astype(int)
    df_std['data_source'] = 'Cardiovascular'
    df_std['collection_date'] = pd.Timestamp('2013-01-01')
    return df_std


def standardize_diabetes_indicators(df):
    """Map Diabetes Health Indicators to common schema"""
    target_col = 'Diabetes_binary' if 'Diabetes_binary' in df.columns else 'Diabetes_012'

    df_std = pd.DataFrame()
    df_std['patient_id'] = [f"DI_{i}" for i in range(len(df))]
    df_std['age'] = (df['Age'] * 5).astype(int)
    df_std['sex'] = df['Sex'].map({1: 'Female', 2: 'Male'})
    df_std['bmi'] = df['BMI'].astype(float) if 'BMI' in df.columns else np.nan
    df_std['physical_activity'] = df['PhysActivity'].astype(int)
    df_std['medical_history_diabetes'] = df[target_col].astype(int)
    df_std['medical_history_hypertension'] = df['HighBP'].astype(int)
    df_std['target_disease'] = df[target_col].astype(int)
    df_std['data_source'] = 'Diabetes_Indicators'
    df_std['collection_date'] = pd.Timestamp('2015-01-01')
    return df_std

print("Standardization functions defined")

Standardization functions defined


## 6. Apply Standardization

In [11]:
standardized_datasets = {}

if 'heart_indicators' in datasets:
    standardized_datasets['heart_indicators'] = standardize_heart_indicators(datasets['heart_indicators'])
    print(f"✓ Heart Indicators: {standardized_datasets['heart_indicators'].shape}")

if 'cardiovascular' in datasets:
    standardized_datasets['cardiovascular'] = standardize_cardiovascular(datasets['cardiovascular'])
    print(f"✓ Cardiovascular: {standardized_datasets['cardiovascular'].shape}")

if 'diabetes_indicators' in datasets:
    standardized_datasets['diabetes_indicators'] = standardize_diabetes_indicators(datasets['diabetes_indicators'])
    print(f"✓ Diabetes Indicators: {standardized_datasets['diabetes_indicators'].shape}")

✓ Heart Indicators: (253680, 14)
✓ Cardiovascular: (70000, 13)
✓ Diabetes Indicators: (253680, 10)


## 7. Merge All Datasets

In [12]:
integrated_df = pd.concat(standardized_datasets.values(), ignore_index=True)

print(f"✓ Integrated Dataset Shape: {integrated_df.shape}")
print(f"✓ Total Records: {len(integrated_df):,}")
print(f"\nData Source Distribution:")
print(integrated_df['data_source'].value_counts())

✓ Integrated Dataset Shape: (577360, 18)
✓ Total Records: 577,360

Data Source Distribution:
data_source
Heart_Indicators       253680
Diabetes_Indicators    253680
Cardiovascular          70000
Name: count, dtype: int64


## 8. Handle Outliers (IQR Method)

In [13]:
def detect_outliers_iqr(df, columns, multiplier=1.5):
    """Detect outliers using IQR method"""
    outliers_mask = pd.DataFrame(False, index=df.index, columns=columns)
    
    for col in columns:
        if df[col].dtype in ['float64', 'int64']:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - multiplier * IQR
            upper_bound = Q3 + multiplier * IQR
            outliers_mask[col] = (df[col] < lower_bound) | (df[col] > upper_bound)
    
    return outliers_mask

# Detect outliers in numeric columns
numeric_cols = ['age', 'bmi', 'blood_pressure_systolic', 'blood_pressure_diastolic', 
                'cholesterol', 'glucose']
numeric_cols = [col for col in numeric_cols if col in integrated_df.columns]

outlier_mask = detect_outliers_iqr(integrated_df, numeric_cols)
num_outliers = outlier_mask.sum().sum()

print(f"✓ Detected {num_outliers} outlier instances")
print(f"\nOutliers by Feature:")
print(outlier_mask.sum())

✓ Detected 36282 outlier instances

Outliers by Feature:
age                             0
bmi                         19694
blood_pressure_systolic      1435
blood_pressure_diastolic     4632
cholesterol                     0
glucose                     10521
dtype: int64


## 9. Handle Missing Values

In [16]:
print("Missing value imputation strategy:")
print("1. Categorical: Mode imputation")
print("2. Numeric: Batched KNN Imputation (k=5)\n")

# Identify columns
categorical_cols = integrated_df.select_dtypes(include=['object']).columns.tolist()
numeric_cols_all = integrated_df.select_dtypes(include=['float64', 'int64']).columns.tolist()

categorical_cols = [col for col in categorical_cols if col not in ['patient_id', 'data_source']]

# Categorical imputation (Mode)
for col in categorical_cols:
    if integrated_df[col].isnull().sum() > 0:
        mode_vals = integrated_df[col].mode(dropna=True)
        mode_val = mode_vals.iloc[0] if len(mode_vals) > 0 else 'Unknown'
        integrated_df[col] = integrated_df[col].fillna(mode_val)
        print(f"✓ Imputed {col}")

def batched_knn_impute(df, numeric_cols, batch_size=50000, n_neighbors=5):
    """Impute numeric columns in batches and merge the results back."""
    df_imputed = df.copy()

    if 'data_source' in df_imputed.columns:
        source_groups = df_imputed.groupby('data_source', sort=False).groups
    else:
        source_groups = {'all': df_imputed.index}

    for source_name, source_idx in source_groups.items():
        source_df = df_imputed.loc[source_idx].copy()
        source_missing = source_df[numeric_cols].isnull().sum().sum()

        if source_missing == 0:
            continue

        print(f"\nProcessing {source_name} in batches...")

        for start in range(0, len(source_df), batch_size):
            end = min(start + batch_size, len(source_df))
            batch = source_df.iloc[start:end].copy()
            batch_idx = batch.index

            batch_numeric_cols = [col for col in numeric_cols if batch[col].notna().any()]
            if not batch_numeric_cols:
                continue

            if len(batch) < 2:
                continue

            batch_missing = batch[batch_numeric_cols].isnull().sum().sum()
            if batch_missing == 0:
                df_imputed.loc[batch_idx, batch_numeric_cols] = batch[batch_numeric_cols].values
                continue

            imputer = KNNImputer(n_neighbors=min(n_neighbors, len(batch)))
            imputed_values = imputer.fit_transform(batch[batch_numeric_cols])
            df_imputed.loc[batch_idx, batch_numeric_cols] = imputed_values

            print(f"  ✓ Rows {start + 1:,}-{end:,} imputed")

    # Final fallback for any remaining numeric gaps
    remaining_missing = df_imputed[numeric_cols].isnull().sum().sum()
    if remaining_missing > 0:
        print(f"\nFilling remaining {remaining_missing:,} numeric gaps with column medians...")
        for col in numeric_cols:
            if df_imputed[col].isnull().any():
                df_imputed[col] = df_imputed[col].fillna(df_imputed[col].median())

    return df_imputed

# Numeric imputation (batched KNN)
numeric_missing = integrated_df[numeric_cols_all].isnull().sum().sum()
if numeric_missing > 0:
    print(f"\nApplying batched KNN imputation for {numeric_missing:,} numeric values...")
    integrated_df = batched_knn_impute(
        integrated_df,
        numeric_cols_all,
        batch_size=50000,
        n_neighbors=5
    )
    print("✓ Batched KNN Imputation complete")
else:
    print("\n✓ No numeric missing values")

print(f"\nTotal missing values after imputation: {integrated_df.isnull().sum().sum()}")

Missing value imputation strategy:
1. Categorical: Mode imputation
2. Numeric: Batched KNN Imputation (k=5)


✓ No numeric missing values

Total missing values after imputation: 70000


## 10. Feature Engineering

In [28]:
print("Missing value imputation strategy:")
print("1. Categorical: Source-wise mode imputation")
print("2. Numeric: Source-wise median imputation")
print("3. Derived feature: Recompute bmi_category from bmi\n")

# Show missing columns before imputation
missing_before = integrated_df.isnull().sum()
missing_before = missing_before[missing_before > 0].sort_values(ascending=False)
print("Columns with missing values before imputation:")
print(missing_before if len(missing_before) else "None")
print()

# Columns by type
categorical_cols = integrated_df.select_dtypes(include=['object', 'category']).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in ['patient_id', 'data_source', 'bmi_category']]

numeric_cols_all = integrated_df.select_dtypes(include=['float64', 'int64', 'int32', 'float32']).columns.tolist()
datetime_cols = integrated_df.select_dtypes(include=['datetime64[ns]', 'datetimetz']).columns.tolist()

def fill_categorical_by_source(df, col):
    """Fill categorical/object column with source-wise mode, safely handling category dtype."""
    for source in df['data_source'].dropna().unique():
        source_mask = df['data_source'].eq(source)
        missing_mask = source_mask & df[col].isna()

        if not missing_mask.any():
            continue

        mode_vals = df.loc[source_mask, col].mode(dropna=True)
        fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else 'Unknown'

        # If it's a categorical column, add the new category first if needed
        if pd.api.types.is_categorical_dtype(df[col]) and fill_val not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories([fill_val])

        df.loc[missing_mask, col] = fill_val

    # Final fallback for any remaining missing values
    if df[col].isna().any():
        mode_vals = df[col].mode(dropna=True)
        fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else 'Unknown'

        if pd.api.types.is_categorical_dtype(df[col]) and fill_val not in df[col].cat.categories:
            df[col] = df[col].cat.add_categories([fill_val])

        df[col] = df[col].fillna(fill_val)

# Categorical imputation
for col in categorical_cols:
    if integrated_df[col].isna().any():
        fill_categorical_by_source(integrated_df, col)
        print(f"✓ Imputed {col}")

# Datetime imputation
for col in datetime_cols:
    if integrated_df[col].isna().any():
        for source in integrated_df['data_source'].dropna().unique():
            source_mask = integrated_df['data_source'].eq(source)
            missing_mask = source_mask & integrated_df[col].isna()

            if not missing_mask.any():
                continue

            mode_vals = integrated_df.loc[source_mask, col].mode(dropna=True)
            fill_val = mode_vals.iloc[0] if len(mode_vals) > 0 else pd.Timestamp('2015-01-01')
            integrated_df.loc[missing_mask, col] = fill_val

        if integrated_df[col].isna().any():
            mode_vals = integrated_df[col].mode(dropna=True)
            integrated_df[col] = integrated_df[col].fillna(
                mode_vals.iloc[0] if len(mode_vals) > 0 else pd.Timestamp('2015-01-01')
            )
        print(f"✓ Imputed {col}")

# Numeric imputation
for col in numeric_cols_all:
    if integrated_df[col].isna().any():
        for source in integrated_df['data_source'].dropna().unique():
            source_mask = integrated_df['data_source'].eq(source)
            missing_mask = source_mask & integrated_df[col].isna()

            if not missing_mask.any():
                continue

            median_val = integrated_df.loc[source_mask, col].median()
            integrated_df.loc[missing_mask, col] = median_val

        if integrated_df[col].isna().any():
            integrated_df[col] = integrated_df[col].fillna(integrated_df[col].median())

        print(f"✓ Imputed {col}")

# Recompute bmi_category from bmi after BMI imputation
if 'bmi' in integrated_df.columns:
    integrated_df['bmi_category'] = pd.cut(
        integrated_df['bmi'],
        bins=[0, 18.5, 25, 30, np.inf],
        labels=['Underweight', 'Normal', 'Overweight', 'Obese']
    )

    # Safety fallback if anything is still missing
    if integrated_df['bmi_category'].isna().any():
        if pd.api.types.is_categorical_dtype(integrated_df['bmi_category']):
            if 'Unknown' not in integrated_df['bmi_category'].cat.categories:
                integrated_df['bmi_category'] = integrated_df['bmi_category'].cat.add_categories(['Unknown'])
        integrated_df['bmi_category'] = integrated_df['bmi_category'].fillna('Unknown')

    print("✓ Recomputed bmi_category from bmi")

print(f"\nTotal missing values after imputation: {integrated_df.isnull().sum().sum()}")

print("\nColumns with missing values after imputation:")
missing_after = integrated_df.isnull().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)
print(missing_after if len(missing_after) else "None")

Missing value imputation strategy:
1. Categorical: Source-wise mode imputation
2. Numeric: Source-wise median imputation
3. Derived feature: Recompute bmi_category from bmi

Columns with missing values before imputation:
bmi_category    70000
dtype: int64

✓ Recomputed bmi_category from bmi

Total missing values after imputation: 0

Columns with missing values after imputation:
None


In [29]:
df.isnull().sum()

Diabetes_binary         0
HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
dtype: int64

## 11. Normalization & Scaling

In [32]:
# Work on a fresh copy so rerunning the cell does not re-scale already scaled data
integrated_df_clean = integrated_df.copy()

numeric_features = [
    'age', 'bmi', 'blood_pressure_systolic', 'blood_pressure_diastolic',
    'cholesterol', 'glucose', 'pulse_pressure'
]
numeric_features = [col for col in numeric_features if col in integrated_df_clean.columns]

print("Applying source-wise outlier clipping + StandardScaler...")

# Clip extreme values per data source on the unscaled data
if 'data_source' in integrated_df_clean.columns:
    for source in integrated_df_clean['data_source'].dropna().unique():
        source_mask = integrated_df_clean['data_source'].eq(source)

        for col in numeric_features:
            if integrated_df_clean.loc[source_mask, col].notna().any():
                q1 = integrated_df_clean.loc[source_mask, col].quantile(0.01)
                q99 = integrated_df_clean.loc[source_mask, col].quantile(0.99)

                # If the column is constant in this source, skip clipping
                if pd.isna(q1) or pd.isna(q99) or q1 == q99:
                    continue

                integrated_df_clean.loc[source_mask, col] = integrated_df_clean.loc[source_mask, col].clip(q1, q99)
                print(f"✓ {source} - clipped {col} to [{q1:.3f}, {q99:.3f}]")
else:
    for col in numeric_features:
        if integrated_df_clean[col].notna().any():
            q1 = integrated_df_clean[col].quantile(0.01)
            q99 = integrated_df_clean[col].quantile(0.99)

            if pd.isna(q1) or pd.isna(q99) or q1 == q99:
                continue

            integrated_df_clean[col] = integrated_df_clean[col].clip(q1, q99)
            print(f"✓ Clipped {col} to [{q1:.3f}, {q99:.3f}]")

# Standardize once
scaler = StandardScaler()
integrated_df_clean[numeric_features] = scaler.fit_transform(
    integrated_df_clean[numeric_features].fillna(integrated_df_clean[numeric_features].mean())
)

print("✓ StandardScaler applied")

print("\nScaled Data Statistics:")
print(integrated_df_clean[numeric_features].describe().round(3))

Applying source-wise outlier clipping + StandardScaler...
✓ Heart_Indicators - clipped age to [-2.432, 1.545]
✓ Heart_Indicators - clipped bmi to [-1.786, 3.856]
✓ Cardiovascular - clipped age to [-0.179, 1.478]
✓ Cardiovascular - clipped blood_pressure_systolic to [-4.913, 9.427]
✓ Cardiovascular - clipped blood_pressure_diastolic to [-0.582, 24.653]
✓ Cardiovascular - clipped cholesterol to [-0.168, 7.368]
✓ Cardiovascular - clipped glucose to [-0.129, 9.281]
✓ Cardiovascular - clipped pulse_pressure to [-24.565, 1.281]
✓ Diabetes_Indicators - clipped age to [-2.432, 1.545]
✓ Diabetes_Indicators - clipped bmi to [-1.786, 3.856]
✓ StandardScaler applied

Scaled Data Statistics:
              age         bmi  blood_pressure_systolic  \
count  577360.000  577360.000               577360.000   
mean       -0.000       0.000                    0.000   
std         1.000       1.000                    1.000   
min        -2.432      -1.786                   -4.913   
25%        -0.775     

## 12. Categorical Encoding

In [33]:
# One-Hot Encoding
categorical_features = ['sex', 'smoking_status', 'bmi_category', 'bp_status', 'age_group']
categorical_features = [col for col in categorical_features if col in integrated_df.columns]

# Prefer the cleaned dataframe if it exists
df_for_encoding = integrated_df_clean.copy() if 'integrated_df_clean' in globals() else integrated_df.copy()

print("Applying One-Hot Encoding...")

integrated_df_encoded = pd.get_dummies(
    df_for_encoding,
    columns=categorical_features,
    drop_first=False,
    dtype=int
)

print("✓ One-Hot Encoding applied")
print(f"\nEncoded Dataset Shape: {integrated_df_encoded.shape}")
print(f"New Features Created: {integrated_df_encoded.shape[1] - df_for_encoding.shape[1]}")

Applying One-Hot Encoding...
✓ One-Hot Encoding applied

Encoded Dataset Shape: (577360, 36)
New Features Created: 13


## 13. Data Validation

In [34]:
print("FINAL DATA QUALITY REPORT")
print("=" * 60)

df_source = integrated_df_clean if 'integrated_df_clean' in globals() else integrated_df

print(f"✓ Total Records: {len(df_source):,}")
print(f"✓ Total Features (after encoding): {integrated_df_encoded.shape[1]}")
print(f"✓ Missing Values: {integrated_df_encoded.isnull().sum().sum()}")
print(f"✓ Duplicate Records: {integrated_df_encoded.duplicated().sum()}")
print(f"✓ Data Completeness: {100.0 if integrated_df_encoded.isnull().sum().sum() == 0 else 0.0}%")

print(f"\nData Sources:")
if 'data_source' in df_source.columns:
    for source, count in df_source['data_source'].value_counts().items():
        print(f"  - {source}: {count:,}")
else:
    print("  - data_source column not found")

FINAL DATA QUALITY REPORT
✓ Total Records: 577,360
✓ Total Features (after encoding): 36
✓ Missing Values: 0
✓ Duplicate Records: 0
✓ Data Completeness: 100.0%

Data Sources:
  - Heart_Indicators: 253,680
  - Diabetes_Indicators: 253,680
  - Cardiovascular: 70,000


## 14. Export Processed Datasets

In [35]:
# Prefer cleaned dataframe if available
df_to_export = integrated_df_clean if 'integrated_df_clean' in globals() else integrated_df

# Export raw integrated dataset (standardized, cleaned)
df_to_export.to_csv(OUTPUT_DIR / 'integrated_healthcare_dataset.csv', index=False)
print(f"✓ Saved: integrated_healthcare_dataset.csv ({df_to_export.shape})")

# Export encoded dataset for ML models
integrated_df_encoded.to_csv(OUTPUT_DIR / 'healthcare_dataset_encoded.csv', index=False)
print(f"✓ Saved: healthcare_dataset_encoded.csv ({integrated_df_encoded.shape})")

# Export data dictionary
data_dictionary = pd.DataFrame({
    'Column': df_to_export.columns,
    'Data_Type': df_to_export.dtypes.astype(str),
    'Non_Null_Count': df_to_export.count(),
    'Null_Count': df_to_export.isnull().sum(),
})
data_dictionary.to_csv(OUTPUT_DIR / 'data_dictionary.csv', index=False)
print("✓ Saved: data_dictionary.csv")

print(f"\n✓ All datasets exported to: {OUTPUT_DIR}")

✓ Saved: integrated_healthcare_dataset.csv ((577360, 23))
✓ Saved: healthcare_dataset_encoded.csv ((577360, 36))
✓ Saved: data_dictionary.csv

✓ All datasets exported to: ../Data_preprocessing/output


## 15. Preprocessing Summary

In [22]:
print("\n" + "="*70)
print("PREPROCESSING SUMMARY")
print("="*70)

print("\n✓ PROCESSING PIPELINE:")
print("  1. ✓ Loaded 3 datasets")
print("  2. ✓ Standardized to common schema")
print("  3. ✓ Integrated into single dataset")
print("  4. ✓ Detected outliers (retained for domain relevance)")
print("  5. ✓ Imputed missing values (KNN method)")
print("  6. ✓ Engineered 5 clinical features")
print("  7. ✓ Normalized numeric features")
print("  8. ✓ Encoded categorical variables")
print("  9. ✓ Validated data quality (100% completeness)")
print("  10. ✓ Exported 3 formats")

print("\n✓ OUTPUT DATASETS:")
print(f"  1. integrated_healthcare_dataset.csv")
print(f"     - Shape: {integrated_df.shape}")
print(f"     - Use: Feature analysis, baseline models")
print(f"\n  2. healthcare_dataset_encoded.csv")
print(f"     - Shape: {integrated_df_encoded.shape}")
print(f"     - Use: ML models (BiLSTM, XGBoost)")
print(f"\n  3. data_dictionary.csv")
print(f"     - Use: Documentation, understanding schema")

print(f"\n✓ QUALITY METRICS:")
print(f"  - Data Completeness: 100%")
print(f"  - Duplicates: 0")
print(f"  - Missing Values: 0")
print(f"  - Outliers Treated: {num_outliers} instances detected (retained)")
print(f"  - Features Engineered: 5 clinical + 1 derived")
print(f"  - Total Features (Encoded): {integrated_df_encoded.shape[1]}")
print("\n" + "="*70)
print("✓ PREPROCESSING COMPLETE - Ready for Modeling!")
print("="*70)


PREPROCESSING SUMMARY

✓ PROCESSING PIPELINE:
  1. ✓ Loaded 3 datasets
  2. ✓ Standardized to common schema
  3. ✓ Integrated into single dataset
  4. ✓ Detected outliers (retained for domain relevance)
  5. ✓ Imputed missing values (KNN method)
  6. ✓ Engineered 5 clinical features
  7. ✓ Normalized numeric features
  8. ✓ Encoded categorical variables
  9. ✓ Validated data quality (100% completeness)
  10. ✓ Exported 3 formats

✓ OUTPUT DATASETS:
  1. integrated_healthcare_dataset.csv
     - Shape: (577360, 23)
     - Use: Feature analysis, baseline models

  2. healthcare_dataset_encoded.csv
     - Shape: (577360, 35)
     - Use: ML models (BiLSTM, XGBoost)

  3. data_dictionary.csv
     - Use: Documentation, understanding schema

✓ QUALITY METRICS:
  - Data Completeness: 100%
  - Duplicates: 0
  - Missing Values: 0
  - Outliers Treated: 36282 instances detected (retained)
  - Features Engineered: 5 clinical + 1 derived
  - Total Features (Encoded): 35

✓ PREPROCESSING COMPLETE - R